In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.metrics import f1_score
import time
import os
import pickle


# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)


def load_trace_data(csv_file, property_file, prefix_extraction=True, min_prefix_length=2, max_trace_length=50, return_case_ids=False):
    df = pd.read_csv(csv_file)
    df['start_time'] = pd.to_datetime(df['start_time'])
    df = df.sort_values(['case:concept:name', 'start_time'])
    property_dict = load_pickle(property_file)
    traces, labels = [], []
    case_ids = [] if return_case_ids else None
    activity_to_idx = {}
    idx_counter = 1

    for case_id, group in tqdm(df.groupby('case:concept:name'), desc="Extracting prefixes"):
        submitted_time = None
        sent_time = None
        seq = []
        prev_time = None
        prop_values = property_dict[case_id]

        for _, row in group.iterrows():
            event_name = row['concept:name']
            timestamp = row['start_time']

            if event_name not in activity_to_idx:
                activity_to_idx[event_name] = idx_counter
                idx_counter += 1

            delta = 0 if prev_time is None else (timestamp - prev_time).total_seconds()
            prev_time = timestamp
            seq.append([activity_to_idx[event_name], delta])

            if prefix_extraction and len(seq) >= min_prefix_length:
                diff_time = (sent_time - submitted_time).total_seconds() if (submitted_time and sent_time) else -1
               
                traces.append(torch.tensor(seq[:], dtype=torch.float32))
                labels.append(prop_values)
                if return_case_ids:
                    case_ids.append(case_id)

    traces_tensor = pad_sequence(traces, batch_first=True, padding_value=0)
    labels_tensor = torch.tensor(labels, dtype=torch.float32)
    if return_case_ids:
        return traces_tensor, labels_tensor, activity_to_idx, case_ids
    else:
        return traces_tensor, labels_tensor, activity_to_idx


train_traces, train_labels_prop, act2idx = load_trace_data('remove_partial/train_set.csv','remove_partial/train_prop.pickle', prefix_extraction=True, min_prefix_length=2)
test_traces, test_labels_prop, _ = load_trace_data('remove_partial/test_set.csv','remove_partial/test_prop.pickle', prefix_extraction=True, min_prefix_length=2)

Device in use: cpu


Extracting prefixes: 100%|████████████████| 3821/3821 [00:02<00:00, 1752.81it/s]


In [5]:
train_labels = 1. * (train_labels_prop > 0) 
test_labels =1. * (test_labels_prop > 0) 

train_labels = train_labels[:,:]
test_labels = test_labels[:,:]

In [7]:
n = 30000
train_dataset = TensorDataset(train_traces[:n], train_labels[:n])
test_dataset = TensorDataset(test_traces[:n], test_labels[:n])

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --------- LSTM with Attention Model ----------
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim, 1)  # Attention scores
        self.softmax = nn.Softmax(dim=1)  # Normalize over time steps

    def forward(self, lstm_out):
        attn_scores = self.attn(lstm_out)  # (batch, seq_len, 1)
        attn_weights = self.softmax(attn_scores)  # (batch, seq_len, 1)
        context_vector = torch.sum(attn_weights * lstm_out, dim=1)  # (batch, hidden_dim)
        return context_vector, attn_weights

class LSTMWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2):
        super(LSTMWithAttention, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # LSTM Layer
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)

        # Attention Layer
        self.attention = Attention(hidden_dim)

        # Fully connected output layer (3 outputs)
        self.fc = nn.Linear(hidden_dim, 3)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden_dim)
        
        # Apply attention
        context_vector, attn_weights = self.attention(lstm_out)

        # Fully connected layer to output 3 values
        output = self.fc(context_vector)  # (batch, 3)

        # Apply custom activation to allow ±∞ values
        output = torch.tanh(output) * 1e10  # Scale to allow very large/small values
        
        return output, attn_weights

# --------- Generate Synthetic Data ----------
def generate_data(num_samples=1000, seq_length=5, input_dim=10):
    X = np.random.randn(num_samples, seq_length, input_dim).astype(np.float32)  # Random sequences
    y = np.random.randn(num_samples, 3).astype(np.float32)  # Target values
    y[np.random.rand(*y.shape) > 0.8] = np.inf  # Add ±∞ values randomly
    y[np.random.rand(*y.shape) > 0.9] = -np.inf
    return torch.tensor(X), torch.tensor(y)

# --------- Training Setup ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
input_dim = 10
hidden_dim = 32
num_layers = 2
learning_rate = 0.001
batch_size = 32
epochs = 20
patience = 5  # Early stopping

# Generate dataset

# Initialize model
model = LSTMWithAttention(input_dim, hidden_dim, num_layers).to(device)
criterion = nn.MSELoss()  # Loss function
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)  # L2 Regularization
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

# --------- Training Loop ----------
best_val_loss = float("inf")
early_stop_counter = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Handle Inf values by replacing with large finite numbers
        batch_y = torch.where(torch.isinf(batch_y), torch.sign(batch_y) * 1e10, batch_y)

        optimizer.zero_grad()
        output, _ = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation Loop
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            batch_y = torch.where(torch.isinf(batch_y), torch.sign(batch_y) * 1e10, batch_y)
            output, _ = model(batch_X)
            loss = criterion(output, batch_y)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    # Learning Rate Scheduler
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pth")  # Save best model
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping triggered!")
            break

print("Training Complete!")

# --------- Load Best Model ----------
model.load_state_dict(torch.load("best_model.pth"))
model.eval()


/home/simones/miniconda3/envs/torch/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


RuntimeError: input.size(-1) must be equal to input_size. Expected 10, got 2